In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import polars as pl
import numpy as np
from deepctr.feature_column import SparseFeat, VarLenSparseFeat
from preprocess import gen_data_set, gen_model_input
from sklearn.preprocessing import LabelEncoder
from tensorflow.python.keras import backend as K
from tensorflow.python.keras.models import Model

from deepmatch.models import *
from deepmatch.utils import sampledsoftmaxloss, NegativeSampler
import sys

sys.path.append("../src/otto_recommender")
import utils
import deepmatch_preprocess

2023-01-22 19:59:00.140747: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-01-22 19:59:00.262936: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2023-01-22 19:59:00.262968: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2023-01-22 19:59:00.953275: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2023-

INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [2]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch13_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    new_split = ""


config = Config()

In [3]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, new_split=False)
# X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
#     config, 
#     X_train, 
#     X_val, 
#     X_val_seq, 
#     ordinal_encoder, 
#     return_only_val=True, 
#     truncate_val=True
# )
pad = ordinal_encoder.unknown_value

# X_train = X_train.drop("count")
# X_train = X_train.drop("n_session_right")
# X_val = X_val.drop("count")
# X_val_seq = X_val_seq.drop("count")

CPU times: user 15min 30s, sys: 1min 51s, total: 17min 22s
Wall time: 3min 42s


In [4]:
X_train.head()

user,aid,ts,type,n_session,num_action
i64,i64,i64,str,i32,i32
14250687,467903,1662328738422,"""clicks""",2,4
14250687,467903,1662328710275,"""clicks""",2,5
14571534,139680,1662328709214,"""clicks""",0,4
14571149,278804,1662328707619,"""clicks""",0,4
14250687,467903,1662328701548,"""clicks""",2,6


In [8]:
data = X_train.to_pandas()

In [6]:
col_ts = config.col_ts
col_item = config.col_aid
col_type = config.col_type
col_user = config.col_user

In [10]:
data.sort_values(col_ts, inplace=True)

In [11]:
%%time

item_ids = data[col_item].unique()
item_id_types_map = dict(zip(data[col_item].values, data[col_type].values))
train_set = list()
test_set = list()

CPU times: user 1min 3s, sys: 779 ms, total: 1min 3s
Wall time: 1min 3s


In [14]:
item_id_types_map[954978]

'carts'

In [5]:
%%time

sparse_features = [
    "user",
    "aid",
    "type",
]

SEQ_LEN = 10
negsample = 0

feature_max_idx = dict()
feature_lbe = dict()
for feature in sparse_features:
    lbe = LabelEncoder()
    data[feature] = lbe.fit_transform(data[feature]) + 1
    feature_max_idx[feature] = data[feature].max() + 1
    feature_lbe[feature] = lbe

CPU times: user 635 ms, sys: 0 ns, total: 635 ms
Wall time: 634 ms


In [6]:
feature_max_idx

{'movie_id': 3707,
 'user_id': 6041,
 'gender': 3,
 'age': 8,
 'occupation': 22,
 'zip': 3440,
 'genres': 19}

In [7]:
%%time

user_profile = data[["user", "aid", "type"]].drop_duplicates("user")

item_profile = data[["movie_id", "genres"]].drop_duplicates("movie_id")

user_profile.set_index("user_id", inplace=True)

user_item_list = data.groupby("user_id")["movie_id"].apply(list)

train_set, test_set = gen_data_set(data, SEQ_LEN, negsample)

train_model_input, train_label = gen_model_input(train_set, user_profile, SEQ_LEN)
test_model_input, test_label = gen_model_input(test_set, user_profile, SEQ_LEN)

100%|██████████████████████████████████████| 6040/6040 [00:06<00:00, 885.85it/s]


8 8


/media/mikhail/disk/ML/compet/kaggle/otto-recommender-system/otto-recommender/notebooks/preprocess.py:86: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  train_seq_genres = np.array([line[5] for line in train_set])


CPU times: user 15.3 s, sys: 420 ms, total: 15.7 s
Wall time: 15.7 s


In [11]:
test_set

[(3299,
  2966,
  1,
  [2796, 2945, 2739, 2651, 3365, 2335, 1445, 1768, 1980, 971],
  10,
  [7, 8, 5, 3, 12, 1, 8, 12, 11, 12],
  7,
  5),
 (4016,
  3551,
  1,
  [2220, 1494, 2219, 199, 2218, 466, 2341, 2778, 2336, 2978],
  10,
  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
  1,
  3),
 (1029,
  3267,
  1,
  [2355, 1237, 147, 2447, 1980, 1815, 1167, 1251, 1187, 1130],
  10,
  [11, 11, 11, 11, 11, 11, 11, 11, 5, 11],
  11,
  2),
 (4590,
  594,
  1,
  [1486, 1018, 2079, 50, 1030, 1199, 1965, 1122, 2059, 1216],
  10,
  [6, 6, 6, 6, 1, 8, 5, 5, 5, 5],
  6,
  4),
 (883,
  2957,
  1,
  [3100, 2909, 628, 3511, 3249, 3322, 3280, 3219, 107, 1980],
  10,
  [5, 5, 1, 1, 5, 1, 8, 5, 1, 11],
  8,
  3),
 (2139,
  139,
  1,
  [895, 103, 1159, 107, 842, 2985, 2261, 2794, 3034, 2710],
  10,
  [5, 8, 8, 1, 8, 16, 8, 16, 1, 6],
  5,
  4),
 (3749,
  2060,
  1,
  [1709, 573, 584, 1304, 3031, 1553, 3041, 742, 225, 3234],
  10,
  [5, 4, 5, 3, 5, 5, 5, 5, 5, 5],
  5,
  3),
 (3889,
  1464,
  1,
  [2587, 908, 1117, 1212, 297

In [10]:
train_set

[(4650,
  2502,
  1,
  [1477, 2587, 2588, 1953, 335, 19, 1051, 1388, 1892, 3278],
  10,
  [1, 5, 5, 2, 5, 5, 1, 5, 5, 8],
  5,
  5),
 (1712,
  2882,
  1,
  [2893, 2129, 1964, 2710, 2047, 1667, 1744, 3494, 2369, 2199],
  10,
  [5, 5, 5, 6, 8, 5, 5, 6, 12, 6],
  5,
  4),
 (5854,
  3257,
  1,
  [633, 2064, 59, 2, 309, 358, 2776, 3154, 2204, 2203],
  10,
  [1, 1, 2, 2, 4, 5, 2, 1, 5, 5],
  2,
  5),
 (238,
  266,
  1,
  [1119, 1197, 2807, 2698, 864, 1261, 194, 2883, 1770, 3105],
  10,
  [8, 5, 6, 8, 8, 8, 8, 8, 8, 8],
  8,
  3),
 (4808,
  444,
  1,
  [1109, 1200, 1289, 1840, 2841, 176, 765, 1363, 1405, 786],
  10,
  [1, 1, 1, 1, 3, 1, 4, 4, 1, 4],
  1,
  3),
 (3934,
  1508,
  1,
  [1228, 1791, 267, 1234, 3595, 2088, 2513, 2692, 1571, 822],
  10,
  [11, 11, 8, 5, 11, 11, 5, 11, 11, 11],
  11,
  2),
 (5598,
  1051,
  1,
  [3460, 1994, 3319, 1782, 2593, 2711, 1815, 1200, 2512, 2587],
  10,
  [6, 5, 5, 8, 5, 5, 11, 1, 5, 5],
  1,
  4),
 (3038,
  854,
  1,
  [964, 859, 2651, 839, 3430, 1131, 234

In [8]:
%%time

embedding_dim = 32

user_feature_columns = [SparseFeat('user_id', feature_max_idx['user_id'], 16),
                        SparseFeat("gender", feature_max_idx['gender'], 16),
                        SparseFeat("age", feature_max_idx['age'], 16),
                        SparseFeat("occupation", feature_max_idx['occupation'], 16),
                        SparseFeat("zip", feature_max_idx['zip'], 16),
                        VarLenSparseFeat(SparseFeat('hist_movie_id', feature_max_idx['movie_id'], embedding_dim,
                                                    embedding_name="movie_id"), SEQ_LEN, 'mean', 'hist_len'),
                        VarLenSparseFeat(SparseFeat('hist_genres', feature_max_idx['genres'], embedding_dim,
                                embedding_name="genres"), SEQ_LEN, 'mean', 'hist_len'),
                        ]

item_feature_columns = [SparseFeat('movie_id', feature_max_idx['movie_id'], embedding_dim),
                        SparseFeat('genres', feature_max_idx['genres'], embedding_dim)
                       ]

from collections import Counter
train_counter = Counter(train_model_input['movie_id'])
item_count = [train_counter.get(i,0) for i in range(item_feature_columns[0].vocabulary_size)]
sampler_config = NegativeSampler('inbatch', num_sampled=255, item_name="movie_id", item_count=item_count)

CPU times: user 149 ms, sys: 108 µs, total: 149 ms
Wall time: 149 ms


In [9]:
%%time

import tensorflow as tf
if tf.__version__ >= '2.0.0':
    tf.compat.v1.disable_eager_execution()
else:
    K.set_learning_phase(True)
    
model = DSSM(user_feature_columns, item_feature_columns,user_dnn_hidden_units=(128,64, embedding_dim),
             item_dnn_hidden_units=(64, embedding_dim,),loss_type='softmax',sampler_config=sampler_config)

model.compile(optimizer="adam", loss=sampledsoftmaxloss)

history = model.fit(train_model_input, train_label,  # train_label,
                    batch_size=256, epochs=3, verbose=1, validation_split=0.0, )

# 4. Generate user features for testing and full item features for retrieval
test_user_model_input = test_model_input
all_item_model_input = {"movie_id": item_profile['movie_id'].values,"genres": item_profile['genres'].values}

user_embedding_model = Model(inputs=model.user_input, outputs=model.user_embedding)
item_embedding_model = Model(inputs=model.item_input, outputs=model.item_embedding)

user_embs = user_embedding_model.predict(test_user_model_input, batch_size=2 ** 12)
# user_embs = user_embs[:, i, :]  # i in [0,k_max) if MIND
item_embs = item_embedding_model.predict(all_item_model_input, batch_size=2 ** 12)

print(user_embs.shape)
print(item_embs.shape)

2023-01-21 22:48:15.318831: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2023-01-21 22:48:15.318899: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory
2023-01-21 22:48:15.318938: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublasLt.so.11'; dlerror: libcublasLt.so.11: cannot open shared object file: No such file or directory
2023-01-21 22:48:15.320342: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcusolver.so.11'; dlerror: libcusolver.so.11: cannot open shared object file: No such file or directory
2023-01-21 22:48:15.320383: W te

Train on 988129 samples
Epoch 1/3


2023-01-21 22:48:16.201578: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-01-21 22:48:16.230548: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:357] MLIR V1 optimization pass is not enabled
2023-01-21 22:48:16.279671: W tensorflow/c/c_api.cc:291] Operation '{name:'training/Adam/dnn/kernel1/v/Assign' id:1372 op device:{requested: '', assigned: ''} def:{{{node training/Adam/dnn/kernel1/v/Assign}} = AssignVariableOp[_has_manual_control_dependencies=true, dtype=DT_FLOAT, validate_shape=false](training/Adam/dnn/kernel1/v, training/Adam/dnn/kernel1/v/Initializer/zeros)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Ei

988129/988129 [==============================] - 41s 42us/sample - loss: 4.4139
Epoch 2/3
988129/988129 [==============================] - 39s 40us/sample - loss: 3.9814
Epoch 3/3
988129/988129 [==============================] - 40s 40us/sample - loss: 3.8434
(6040, 32)
(3706, 32)
CPU times: user 7min 48s, sys: 1min 32s, total: 9min 21s
Wall time: 2min 2s


/media/mikhail/disk/ML/compet/kaggle/otto-recommender-system/otto_env/lib/python3.7/site-packages/tensorflow/python/keras/engine/training.py:2455: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '
2023-01-21 22:50:16.980779: W tensorflow/c/c_api.cc:291] Operation '{name:'lambda/l2_normalize' id:312 op device:{requested: '', assigned: ''} def:{{{node lambda/l2_normalize}} = Mul[T=DT_FLOAT, _has_manual_control_dependencies=true](dnn/dropout_2/cond/Identity, lambda/l2_normalize/Rsqrt)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
2023-01-21 22:50:17.063196: W tensorflow/c/c_api.cc:291] Operation '{name:'lambda_1/l2_normalize' id:408 op 